# ZizAIの環境構築

VS CodeでZizAIのルートを開き、このNotebookにローカルのPython 3.12以降のカーネルを選びます。最初に使うカーネルは、環境構築のセルを実行するためのものです。

このNotebookで行うことは、仮想環境の準備、同じ環境へのZizAIと依存ライブラリの導入、コネクタ共通フォルダーの登録、フォーム用マジックの読み込みです。**「Run All」ではなく番号順にセルを実行**し、③と④の間でカーネルを切り替えてください。

## 0. インストール先と使うコネクタを選ぶ

既定ではZizAI直下の `.venv` に、フォームとCSV用の依存を導入します。すでに選択中のPythonを使いたい場合は `use_project_venv = False` とします。その場合、②と③は選択中の環境にインストール・登録するので、セルが表示するPythonを確認してください。

`installer` は `"pip"` または `"uv"` のどちらかです。uvを選ぶ場合は先にローカルに導入してください。[uv公式の導入手順](https://docs.astral.sh/uv/getting-started/installation/)

必要に応じて `connector_extras` に `excel`、`duckdb`、`bigquery`、`dataintegration`、`python`、`shell`、`vector`、`windows` を追加します。使いたい共通フォルダーが別にある場合は `shared_connectors` で指定します。

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import tomllib

project_root = None              # 自動判定できないときは例: "C:/tools/ZizAI"
shared_connectors = None         # 省略時は ZizAI/connectors
use_project_venv = True
installer = "pip"                # または "uv"
connector_extras = ["csv"]       # 例: ["csv", "excel", "duckdb"]

if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12以降のカーネルを選択してください。")

start = Path(project_root or Path.cwd()).expanduser().resolve()
roots = (start,) if project_root else (start, *start.parents)
root = next((p for p in roots if (p / "pyproject.toml").is_file()
             and (p / "src/forms.py").is_file() and (p / "connectors").is_dir()), None)
if root is None:
    raise ValueError("ZizAIのルートを開くか、project_rootにルートのパスを指定してください。")

catalog = Path(shared_connectors or root / "connectors").expanduser().resolve()
if not catalog.is_dir() or not any(catalog.glob("*/*_definition.yaml")):
    raise ValueError("コネクタYAMLがある共通フォルダーを指定してください。")

project = tomllib.loads((root / "pyproject.toml").read_text(encoding="utf-8"))["project"]
extras = list(dict.fromkeys(["forms", *connector_extras]))
unknown = set(extras) - project["optional-dependencies"].keys()
if unknown:
    raise ValueError(f"不明な追加ライブラリ指定: {', '.join(sorted(unknown))}")
if installer not in {"pip", "uv"}:
    raise ValueError('installerは "pip" または "uv" にしてください。')

environment = root / ".venv" if use_project_venv else Path(sys.prefix)
python = environment / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not use_project_venv:
    python = Path(sys.executable)

def run_target(*args, **kwargs):
    return subprocess.run([str(python), "-I", *args], check=True, **kwargs)

print(f"ZizAI: {root}")
print(f"導入先のPython: {python}")
print(f"共通コネクタ: {catalog}")
print(f"インストーラー: {installer} / 追加ライブラリ: {', '.join(extras)}")

## ① 仮想環境を準備する

`.venv` がなければ作成します。既存の `.venv` を消去・作り直すことはありません。既存カーネルを利用する設定では、仮想環境作成をスキップします。

In [ ]:
if installer == "uv" and not shutil.which("uv"):
    raise RuntimeError('uvが見つかりません。先に導入するか、0で installer = "pip" にしてください。')

if use_project_venv:
    if environment.is_symlink() or environment.is_junction():
        raise ValueError(".venvがリンクになっています。対象のカーネルを明示的に選んでください。")
    if not environment.exists():
        subprocess.run([sys.executable, "-I", "-m", "venv", str(environment)], check=True)
    if not (environment / "pyvenv.cfg").is_file() or not python.is_file():
        raise ValueError("既存の.venvが有効な仮想環境ではありません。内容を確認してください。")

run_target("-c", "from pathlib import Path; import sys; "
           "assert sys.version_info >= (3, 12), 'Python 3.12以上が必要です。'; "
           "assert Path(sys.prefix).resolve() == Path(sys.argv[1]).resolve(), sys.prefix",
           str(environment))
print(f"準備済み: {python}")

## ② ZizAIとライブラリを導入する

①で確認したPythonへ `ipykernel`、ZizAI本体、選択した追加ライブラリを導入します。通常はパッケージ取得に通信が必要です。Vectorのモデル取得やBigQueryへの接続は、ここでは行いません。

In [ ]:
target = f"{root}[{','.join(extras)}]"
if installer == "uv":
    uv = shutil.which("uv")
    if uv is None:
        raise RuntimeError("uvが見つかりません。セル0の設定を確認してください。")
    subprocess.run([uv, "pip", "install", "--python", str(python), "ipykernel", target], check=True)
else:
    run_target("-m", "pip", "--isolated", "install", "ipykernel", target)

run_target("-c", "import ipykernel, importlib.metadata as m; "
           "print('ZizAI:', m.version('zizai-runtime'))")

## ③ コネクタ共通フォルダーを登録する

選んだPython環境に共通フォルダーの場所を登録します。既存の登録先を切り替える場合は、セル0の表示を確認してから実行してください。登録が終わったら次の案内に従ってカーネルを選び直します。

In [ ]:
registration_env = os.environ.copy()
for name in ("ZIZAI_CONFIG_FILE", "ZIZAI_CONNECTORS_DIR"):
    registration_env.pop(name, None)

run_target("-c", "import sys; from zizai.connectors import configure_connectors; "
           "print('登録先:', configure_connectors(sys.argv[1]))",
           catalog.as_posix(), env=registration_env)
print(f"次にVS Codeで選ぶPython: {python}")

## カーネルを選び直す

1. VS CodeのNotebook右上で「カーネルの選択」から、③に表示されたPythonを選びます。`.venv` を使う場合はそのPython環境です。
2. すでにその環境を使っていた場合も、③のあとに**カーネルを再起動**します。Pythonが起動時にコネクタ登録を読み直します。
3. 新しいカーネルで**セル0だけを実行し直し、④へ進みます**。①〜③の再実行は不要です。既存カーネルを使う設定の場合は、③と同じ環境を選んだまま再起動します。

VS Code以外で一覧に表示されない場合は、IPythonのカーネル登録手順を参照してください。[IPythonの公式手順](https://ipython.readthedocs.io/en/stable/install/kernel_install.html)

## ④ マジックを読み込み、フォームを表示する

ここで使うマジックは、通常のNotebookでもカーネルを再起動するたびに読み込みます。フォームで生成したコードは、必要なときに別のセルへコピーして使います。

In [ ]:
if "environment" not in globals():
    raise RuntimeError("カーネル切り替え後にセル0を実行してください。")
if Path(sys.prefix).resolve() != environment.resolve():
    raise RuntimeError(f"③で表示されたPythonをカーネルに選び直してください: {python}")

from zizai.core.connector_config import get_connector_directory
from zizai.form_definition import load_definition

if get_connector_directory() != catalog:
    raise RuntimeError("③で登録したコネクタの場所と異なります。登録先とZIZAI_CONFIG_FILE・ZIZAI_CONNECTORS_DIRを確認してください。")

definitions = load_definition()
print(f"使用中のPython: {sys.executable}")
print("コネクタ登録を確認しました。下の2セルを実行してください。")

In [ ]:
%load_ext zizai.forms

In [ ]:
%form

## Python・Kiroから直接使う場合

同じPython環境で、例えば `import csv_connector` として使えます。マジックの読み込みは、Notebookでフォームを表示する場合に必要です。[導入手順](../docs/usage.md)・[Kiro向けガイド](../.kiro/steering/zizai.md)。